# ⚖️ CourtLLM GRPO Training — HF GPU Version (A10G / L4 / A100)

**Runtime:** 24GB+ GPU (A10G, L4, or A100) | **Estimated time:** 2-3 hours

Trains an 8B LLM to reduce hallucinations using a multi-agent courtroom environment.

## Cell 1: Install Dependencies

In [ ]:
!pip install -q openenv-core unsloth trl transformers sentence-transformers wikipedia-api httpx matplotlib wandb datasets

## Cell 2: Clone CourtLLM from HuggingFace & Set Up Path

In [ ]:
import os, sys

# Clone the Space repo
if not os.path.exists('CourtLLM_OpenEnv'):
    !git clone https://huggingface.co/spaces/mishatul/CourtLLM_OpenEnv CourtLLM_OpenEnv

# Add root to path so 'from client import ...' and 'from models import ...' work
REPO_ROOT = os.path.abspath('CourtLLM_OpenEnv')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Repo cloned and path set!')

## Cell 3: Load 8B Model with Unsloth 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

# Using Llama 3 8B for high-quality reasoning
MODEL_NAME = "unsloth/llama-3-8b-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.float16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    use_gradient_checkpointing=True,
)

print(f"Model loaded: {MODEL_NAME}")

## Cell 4: Connect to CourtLLM Environment

In [ ]:
from client import CourtLLMClient
from models import CourtAction

# Live HF Space URL
ENV_URL = "https://mishatul-courtllm-openenv.hf.space"

# Test connection
with CourtLLMClient(ENV_URL) as client:
    health = client.health()
    print(f"Environment status: {health}")
    obs = client.reset()
    print(f"\nCase ID: {obs.case_id}")

## Cell 5: Helper Functions

In [ ]:
import re
from typing import List

def obs_to_prompt(obs, tokenizer) -> str:
    evidence_str = "\n".join([
        f"[{s['source_id']}] {s['title']}: {s['snippet']}"
        for s in obs.evidence_corpus[:20]
    ])
    claims_str = "\n".join([
        f"- {c['claim_id']}: {c['claim_text']} (Reason: {c['suspicion_reason']})"
        for c in obs.flagged_claims
    ])
    prompt = f"""You are the Defendant in a legal proceeding about factual accuracy.
Respond to the Plaintiff's query with verifiable evidence.

MANDATORY FORMAT:
<claim>
  <statement>Your factual assertion</statement>
  <source_id>EXACT_SOURCE_ID_FROM_CORPUS</source_id>
  <confidence>0.0-1.0</confidence>
</claim>

Evidence Corpus:
{evidence_str}

Plaintiff's Query: {obs.plaintiff_query}

Flagged Claims:
{claims_str}

Your testimony:"""
    return prompt

def parse_defendant_action(completion: str) -> CourtAction:
    claim_pattern = r'<claim>.*?<statement>(.*?)</statement>.*?<source_id>(.*?)</source_id>.*?<confidence>(.*?)</confidence>.*?</claim>'
    matches = re.findall(claim_pattern, completion, re.DOTALL)
    if not matches:
        return CourtAction(
            action_type="generate_testimony",
            content=completion[:500],
            claim_ids=["claim_000"],
            confidence=0.5,
            source_ids=[]
        )
    statement, source_id, confidence = matches[0]
    try: conf = float(confidence.strip())
    except: conf = 0.5
    return CourtAction(
        action_type="generate_testimony",
        content=statement.strip(),
        claim_ids=["claim_000"],
        confidence=conf,
        source_ids=[source_id.strip()] if source_id.strip() else []
    )

def generate(model, tokenizer, prompt: str, max_length: int = 512) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

## Cell 6: Define Reward Function

In [ ]:
def courtroom_reward_fn(completions: List[str], prompts: List[str] = None, **kwargs) -> List[float]:
    rewards = []
    with CourtLLMClient(ENV_URL) as env:
        for completion in completions:
            try:
                obs = env.reset()
                action = parse_defendant_action(completion)
                result = env.step(action)
                rewards.append(result.reward)
            except Exception as e:
                rewards.append(-0.5)
    return rewards

## Cell 7: Build Training Dataset (250 episodes)

In [ ]:
from datasets import Dataset

def build_dataset(n_episodes: int = 250, stage: int = 0) -> Dataset:
    prompts = []
    with CourtLLMClient(ENV_URL) as env:
        env.set_stage(stage)
        for i in range(n_episodes):
            obs = env.reset()
            prompt = obs_to_prompt(obs, tokenizer)
            prompts.append({"prompt": prompt})
            if (i + 1) % 25 == 0: print(f"Generated {i + 1}/{n_episodes} prompts")
    return Dataset.from_list(prompts)

train_data = build_dataset(n_episodes=250, stage=0)

## Cell 8: GRPO Training (2-3 Hours on A10G/L4/A100)

In [ ]:
from trl import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    output_dir="./courtllm_grpo_8b",
    max_steps=400,            # Increased for better convergence
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,       # Slightly higher LR for 8B
    num_generations=4,
    max_prompt_length=1536,
    max_completion_length=512,
    temperature=0.8,
    logging_steps=5,
    save_steps=100,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[courtroom_reward_fn],
    args=config,
    train_dataset=train_data,
)

trainer.train()

## Cell 9: Plot Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("outputs", exist_ok=True)

log = trainer.state.log_history
steps   = [x["step"]   for x in log if "reward" in x]
rewards = [x["reward"] for x in log if "reward" in x]

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, rewards, label="Episode Reward", linewidth=2, color='#6c63ff')
    plt.axhline(y=0.65, color='g', linestyle='--', label="Target (0.65)")
    plt.xlabel("Training Step", fontsize=12)
    plt.ylabel("Average Episode Reward", fontsize=12)
    plt.title("CourtLLM GRPO Training (Llama-3-8B) — Reward Curve", fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("outputs/reward_curve_8b.png", dpi=150, bbox_inches='tight')
    plt.show()

## Cell 10: Evaluation & Save Model

In [ ]:
# Save trained model
model.save_pretrained("courtllm_trained_8b")
tokenizer.save_pretrained("courtllm_trained_8b")

print("Model and tokenizer saved successfully to 'courtllm_trained_8b' directory.")
print("\nNow download the outputs/ folder and your trained model!")